### Step 1: Install necesscary packages

In [ ]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [ ]:
import sys
import os
from pathlib import Path

# Run this notebook either from the repository root or the dpo/ folder.
# Never depend on a particular user's home-directory location.
work_dir = Path.cwd().resolve()
if (work_dir / "model.py").is_file() and (work_dir / "dpo" / "dpo.ipynb").is_file():
    repo_root = work_dir
elif (work_dir / "dpo.ipynb").is_file() and (work_dir.parent / "model.py").is_file():
    repo_root = work_dir.parent
else:
    raise RuntimeError("Open the notebook from the NanoGPT-Math root or dpo/ directory.")
dpo_dir = repo_root / "dpo"
sys.path.insert(0, str(repo_root))
# Keep Slurm-assigned CUDA_VISIBLE_DEVICES; do not force GPU 1.
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with (repo_root / "sft" / "meta.pkl").open("rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])print(f"Repository root: {repo_root}")
print(f"Selected device: {device}")


### Step 3: Define helper functions

In [3]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [ ]:
ckpt = torch.load(repo_root / "sft" / "gpt.pt", map_location=device, weights_only=True)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()


### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
# Step 5: load and validate preference pairs.
# The supplied pos_neg_pairs.json has only four DEMO examples.
# Replace/extend it with a larger, checked preference dataset for full training.
data_path = dpo_dir / "pos_neg_pairs.json"
if not data_path.exists():
    data_path = dpo_dir / "data" / "pos_neg_pairs.json"
if not data_path.exists():
    raise FileNotFoundError("Missing dpo/pos_neg_pairs.json")

with data_path.open("r", encoding="utf-8") as f:
    raw_pairs = json.load(f)
if not isinstance(raw_pairs, list) or not raw_pairs:
    raise ValueError("Expected a nonempty JSON array of positive/negative pairs")

lines = []
for index, pair in enumerate(raw_pairs):
    if not isinstance(pair, dict) or not all(k in pair for k in ("positive", "negative")):
        raise ValueError(f"Sample {index} must contain positive and negative strings")
    pos, neg = pair["positive"], pair["negative"]
    if not isinstance(pos, str) or not isinstance(neg, str) or pos == neg:
        raise ValueError(f"Invalid responses at sample {index}")

    # The question ends with '=?' in the course's arithmetic/algebra format.
    # This is more reliable than finding the longest common prefix: two numeric
    # answers can share initial digits, which would mask away answer tokens.
    end = pos.find("=?")
    if end < 0:
        raise ValueError(f"Sample {index}: missing '=?' in positive response")
    prompt = pos[:end + 2]
    if not neg.startswith(prompt):
        raise ValueError(f"Sample {index}: positive/negative prompts do not match")
    for response in (pos, neg):
        missing = sorted(set(response) - set(stoi))
        if missing:
            raise ValueError(f"Sample {index} contains unknown characters: {missing}")
    lines.append({"positive": pos, "negative": neg, "prompt": prompt})

max_pair_length = max(max(len(pair["positive"]), len(pair["negative"])) for pair in lines)
# The course get_batches() appends four newline padding markers before packing.
required_length = max_pair_length + 4
if required_length > gpt.config.block_size:
    raise ValueError(
        f"Longest training sequence ({required_length}) exceeds "
        f"model block_size ({gpt.config.block_size})"
    )
max_length = max(max_length, required_length)  # Do not truncate the question.
if len(lines) < batch_size:
    print(f"Only {len(lines)} pairs: shrinking batch_size for a pipeline smoke test.")
    batch_size = len(lines)
print(f"Loaded {len(lines)} preference pairs from {data_path}")
print(f"batch_size={batch_size}, max_length={max_length}")
if len(lines) <= 4:
    print("WARNING: These are the four course examples, NOT a real training dataset.")


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# Step 6: build the optimizer and learning-rate scheduler.
# No separate SFT training or reference model is needed for the course loss.
import math

optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=base_lr,
    weight_decay=0.01,
)

# The course-provided get_batches() skips the final incomplete batch.
steps_per_epoch = len(lines) // batch_size
if steps_per_epoch == 0:
    raise ValueError("No full batch available: reduce batch_size or add preference data.")
if len(lines) % batch_size:
    print(
        f"Note: {len(lines) % batch_size} example(s) per epoch will be skipped "
        "by the instructor's get_batches() helper."
    )
total_steps = epochs * steps_per_epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=base_lr * 0.1
)
print(f"AdamW lr={base_lr}, epochs={epochs}, steps_per_epoch={steps_per_epoch}")


### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
# Step 7: train with the instructor's recommended preference loss.
# EXACT objective from the original Notebook (beta = 0.5 by default):
# loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean()
#        - pos_logprob.mean() * 0.1
#
# This is a lightweight preference objective with a chosen-response
# log-probability bonus, NOT reference-model DPO. No extra SFT is performed.
# Keep Step 3's compute_logprob() and get_batches() unchanged.

random.seed(42)
torch.manual_seed(42)
gpt.train()
training_loss = []

for epoch in range(epochs):
    running_loss = 0.0
    completed_batches = 0
    pbar = tqdm(
        get_batches(lines, batch_size),
        total=steps_per_epoch,
        desc=f"Epoch {epoch + 1}/{epochs}",
    )

    for neg_tensor, pos_tensor in pbar:
        optimizer.zero_grad(set_to_none=True)

        # Use the instructor's original, sequence-average log-probability helper.
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)

        # The instructor's suggested loss (no frozen reference model).
        preference_loss = -F.logsigmoid(
            (pos_logprob - neg_logprob) / beta
        ).mean()
        chosen_bonus = -0.1 * pos_logprob.mean()
        loss = preference_loss + chosen_bonus

        if not torch.isfinite(loss).item():
            raise RuntimeError(f"Non-finite loss at epoch {epoch + 1}")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        completed_batches += 1
        running_loss += loss.item()
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            pref=f"{preference_loss.item():.4f}",
            chosen=f"{chosen_bonus.item():.4f}",
        )

    if completed_batches == 0:
        raise RuntimeError("Training did not process any batch")

    mean_loss = running_loss / completed_batches
    training_loss.append(mean_loss)
    print(f"Epoch {epoch + 1}/{epochs}: mean loss={mean_loss:.4f}")

    ckpt_path = dpo_dir / "dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt["model_args"],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

print("Training completed.")
print("Important: loss alone is not arithmetic/algebra accuracy.")


### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
ckpt_path = dpo_dir / "dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device, weights_only=True)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).to(device)
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################
